# La propagation de labels, ou l'art de juger une image par ses voisins

Nos objectifs de super-détective :

- Recruter un expert : Charger le modèle qu'on a péniblement entraîné au pseudo-labeling pour qu'il nous aide.
- Cartographier le terrain : Utiliser cet expert pour extraire l'ADN de chaque image (ses embeddings).
- Tisser une toile : Construire un graphe où chaque image est un nœud, connecté à ses plus proches voisins.
- Laisser la magie opérer : Regarder les étiquettes de nos 350 images connues se propager à travers la toile pour deviner les autres.
- Comparer les résultats : Est-ce que cette méthode de 'sagesse des foules' est meilleure que de faire confiance à un seul modèle ? Le suspense est à son comble !

## 1. Préparation du terrain : on reprend (presque) les mêmes !
On commence par importer nos outils et préparer notre jeu de données DermaMNIST. On va recréer notre scénario de départ : 350 images étiquetées (50 par classe) et des milliers d'autres qui attendent d'être identifiées.

On commence par les bases : importer les librairies et charger notre dataset, DermaMNIST. C'est un jeu de données d'images de lésions cutanées. Notre mission : les classifier correctement, même avec une poignée de labels.

DermaMNIST fait partie de la famille des datasets MedMNIST, une collection standardisée de datasets d'images médicales, tous au format 28x28 et organisés de manière similaire aux célèbres datasets MNIST. L'objectif de MedMNIST est de faciliter la recherche et la comparaison de modèles d'apprentissage automatique sur des tâches médicales.

Le dataset DermaMNIST est basé sur le HAM10000, une large collection d'images de dermatoscopie. Il contient des images de 7 classes différentes de lésions cutanées. Son intérêt pédagogique est majeur : il permet d'aborder des problèmes de classification d'images médicales avec un dataset de taille raisonnable, tout en simulant des scénarios de faible quantité de données labellisées, parfait pour explorer des techniques comme la pseudo-labellisation.

Les 7 classes de lésions cutanées sont :

0: actinic keratoses and intraepithelial carcinoma
1: basal cell carcinoma
2: benign keratosis-like lesions
3: dermatofibroma
4: melanoma
5: melanocytic nevi
6: vascular lesions

In [ ]:
# charger les packages necéssaires
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, Subset
from torchvision import transforms
import torchvision.models as models
!pip install medmnist
import medmnist
from medmnist import INFO, Evaluator
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from sklearn.semi_supervised import LabelSpreading
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 6.0 MB/s eta 0:00:00


In [ ]:
# Pour la reproductibilité, parce qu'on est des gens sérieux
torch.manual_seed(42)
np.random.seed(42)

## chargement de nos images et séparation des classes en images labelisées et non labelisées

In [ ]:
# Nom du dataset à charger
data_flag = 'dermamnist'
# Récupère les informations spécifiques à ce dataset depuis medmnist
info = INFO[data_flag]

# Ajout de prints pour mieux comprendre le dataset
print(f"Dataset chargé : {data_flag}")

# Extraire le type de tache(classification, regression, ....)
task = info['task']
print(f"Type de tâche : {task}")

# Extraire le nombre de canaux des images
n_channels = info['n_channels']
print(f"Nombre de canaux : {n_channels}")

# Extraire la taille des images
#input_shape = info['input_shape']
#print(f"Taille des images : {input_shape}")

# extraire le nombre de classe pour la classification
n_classes = len(info['label'])
print(f"Nombre de classes : {n_classes}")

# Récupère la classe Python spécifique pour ce dataset
DataClass = getattr(medmnist, info['python_class'])

Dataset chargé : dermamnist
Type de tâche : multi-class
Nombre de canaux : 3
Nombre de classes : 7


In [ ]:
# Chargement des données
data_flag = 'dermamnist'
info = INFO[data_flag]
n_classes = len(info['label'])
n_channels = info['n_channels']
DataClass = getattr(medmnist, info['python_class'])

# Transformations standard
data_transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize(mean=[.5], std=[.5])])

# On charge le jeu d'entraînement complet et le jeu de test
train_dataset = DataClass(split='train', transform=data_transform, download=True)
test_dataset = DataClass(split='test', transform=data_transform, download=True)

# On recrée notre situation de départ : 50 images par classe étiquetées, et le reste en attente
all_indices = list(range(len(train_dataset)))
labels_array = np.array(train_dataset.labels).flatten()

# Sélectionner 50 images par classe
labeled_indices = []
for c in range(n_classes):
    class_indices = np.where(labels_array == c)[0]
    selected = np.random.choice(class_indices, min(50, len(class_indices)), replace=False)
    labeled_indices.extend(selected)

# Les indices non étiquetés sont le reste
unlabeled_indices = list(set(all_indices) - set(labeled_indices))

print(f'Taille totale du jeu d\'entraînement : {len(train_dataset)} images')
print(f'Données étiquetées (nos indics ) : {len(labeled_indices)} images')
print(f'Données non-étiquetées (les mystères à résoudre ) : {len(unlabeled_indices)} images')

100%|██████████| 19.7M/19.7M [00:34<00:00, 575kB/s]


Taille totale du jeu d'entraînement : 7007 images
Données étiquetées (nos indics ) : 350 images
Données non-étiquetées (les mystères à résoudre ) : 6657 images


In [ ]:
# On définit l'architecture de notre CNN.
# ATTENTION : Elle doit être IDENTIQUE à celle du modèle sauvegardé !
device = "cpu"
class SimpleCNN(nn.Module):
    def __init__(self, in_channels, num_classes):
        super(SimpleCNN, self).__init__()
        self.layers1 = nn.Sequential(
            nn.Conv2d(in_channels, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2))
        self.layers2 = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2))
        # Pour correspondre exactement au modèle de P1C3
        self.fc = nn.Linear(7 * 7 * 32, num_classes)

    def forward(self, x, return_features=False):
        out = self.layers1(x)
        out = self.layers2(out)
        out = out.view(out.size(0), -1)
        if return_features:
            return out  # Retourne les features avant la classification (dimension 1568)
        out = self.fc(out)
        return out

model = SimpleCNN(in_channels=n_channels, num_classes=n_classes)
model_path = '/content/dermamnist_ssl_model.pth'

try:
    state_dict = torch.load(model_path, map_location="cpu")
    # Les noms de couches correspondent exactement, on charge tout
    model.load_state_dict(state_dict)
    print(f'✅ Modèle chargé depuis : {model_path}')
except FileNotFoundError:
    print(f'🚨 Oups ! Le fichier {model_path} est introuvable.')
    print('Veuillez d\'abord exécuter le notebook P1C3 pour entraîner et sauvegarder le modèle.')
    raise

# On passe le modèle sur le bon appareil et en mode évaluation
model.to("cpu")
model.eval()

✅ Modèle chargé depuis : /content/dermamnist_ssl_model.pth


SimpleCNN(
  (layers1): Sequential(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (layers2): Sequential(
    (0): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (fc): Linear(in_features=1568, out_features=7, bias=True)
)

## 3. Extraction des 'coordonnées GPS' (embeddings)
Maintenant que notre expert est prêt, on va le faire passer sur toutes les images de notre jeu d'entraînement (étiquetées ou non) pour obtenir leurs fameux embeddings. C'est comme créer une carte d'identité pour chaque image.

In [ ]:
def get_embeddings(model, dataset, device):
  "extraire les embedding d'un dataset en utilisant un modèle donné"
  model.eval()
  embeddings = []
  loader = DataLoader(dataset, batch_size=256, shuffle=False, num_workers=2)

  with torch.no_grad():
    for images, _ in tqdm (loader, desc = 'extraction des embedding'):
      images = images.to(device)
      with torch.no_grad():
        features = model(images, return_features=True)
        embeddings.append(features.cpu().numpy())

    return np.vstack(embeddings)

# On extrait les embeddings en utilisant notre modèle
all_embeddings = get_embeddings(model, train_dataset, "cpu")

print(f'\nExtraction terminée ! On a obtenu {all_embeddings.shape[0]} embeddings de dimension {all_embeddings.shape[1]}.')

extraction des embedding: 100%|██████████| 28/28 [00:03<00:00,  8.28it/s]


Extraction terminée ! On a obtenu 7007 embeddings de dimension 1568.


## 4. La propagation des rumeurs (de labels)
C'est le moment que vous attendiez tous ! On va utiliser l'algorithme ***LabelSpreading de scikit-learn***.

Comment ça marche ?

Il prend tous nos embeddings et construit un graphe de similarité (notre fameuse toile ).
On lui donne les 350 étiquettes qu'on connaît. Pour les autres, on met une étiquette spéciale : -1 (qui veut dire 'Je ne sais pas').
L'algorithme va alors 'propager' l'influence des étiquettes connues à leurs voisins, puis aux voisins de leurs voisins, jusqu'à ce que chaque image ait une étiquette probable.
C'est un processus démocratique où chaque image est influencée par sa communauté !

In [ ]:
# on prepare le tableau des labels pour notre algorithme
labels_for_spreading = np.full(len(train_dataset), -1, dtype=int)
labels_for_spreading[labeled_indices] = labels_array[labeled_indices]

print(f'Verification : {np.sum(labels_for_spreading != -1)} labels sont connus. Parfait !')

# on instancie le model labelspreading
label_spreading_model = LabelSpreading(kernel='knn', n_neighbors=10, n_jobs=-1)

print('Propagation des labels en cours... C\'est le moment d\'aller prendre un café ')
label_spreading_model.fit(all_embeddings, labels_for_spreading) # on entraine le model
print('Propagation terminée ! Voyons ce qu\'on a trouvé.')

# on recupére l'ensemble des labels prédits pour l'ensemble du dataset
predicted_label = label_spreading_model.transduction_

# on recupére les probabilités prédites pour l'AUC
predicted_probs = label_spreading_model.predict_proba(all_embeddings)




Verification : 350 labels sont connus. Parfait !
Propagation des labels en cours... C'est le moment d'aller prendre un café 
Propagation terminée ! Voyons ce qu'on a trouvé.


Le modèle a rempli tous les trous et a attribué une étiquette à chaque image. Mais est-ce que ces prédictions sont bonnes?

Pour le savoir, on va comparer les étiquettes prédites pour les données initialement non-étiquetées avec leurs vraies étiquettes (qu'on avait cachées). C'est l'heure de vérité !

In [ ]:
from torch.utils.data import Dataset

# Créer un dataset personnalisé avec les labels propagés
class PropagatedDataset(Dataset):
    def __init__(self, dataset, labels):
        self.dataset = dataset
        self.labels = labels

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        img, _ = self.dataset[idx]  # Ignore les labels d'origine, img est déjà un tenseur transformé
        label = self.labels[idx]
        return img, label

# Créer le nouveau dataset avec les labels propagés
train_dataset_propagated = PropagatedDataset(train_dataset, predicted_label)

# Créer un DataLoader pour l'entraînement
train_loader_propagated = DataLoader(train_dataset_propagated, batch_size=32, shuffle=True, num_workers=2)

In [ ]:
def train_and_evaluate(model, train_loader, test_loader, optimizer, criterion, epochs=10):
    """
    Entraîne et évalue un modèle. Retourne (AUC, ACC, F1).
    Si des listes globales metrics_auc/metrics_acc/metrics_f1 existent, y ajoute les scores.
    """
    device = next(model.parameters()).device

    for epoch in range(epochs):
        model.train()
        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.squeeze().long().to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

    # Évaluation
    model.eval()
    y_true = torch.tensor([]).to(device)
    y_score_logits = torch.tensor([]).to(device)
    y_score_preds = torch.tensor([]).to(device)
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            y_true = torch.cat((y_true, labels), 0)
            y_score_logits = torch.cat((y_score_logits, outputs), 0)
            preds = torch.argmax(outputs, dim=1)
            y_score_preds = torch.cat((y_score_preds, preds), 0)

    y_true = y_true.squeeze().cpu().numpy()
    y_score_logits = y_score_logits.detach().cpu().numpy()
    y_score_preds = y_score_preds.detach().cpu().numpy()

    evaluator = Evaluator(data_flag, 'test')
    auc, acc = evaluator.evaluate(y_score_logits)
    f1 = f1_score(y_true, y_score_preds, average='macro')

    try:
        metrics_auc.append(auc)
        metrics_acc.append(acc)
        metrics_f1.append(f1)
    except NameError:
        pass

    print(f'AUC: {auc:.3f}, Accuracy: {acc:.3f}, F1: {f1:.3f}')
    return (auc, acc, f1)

In [ ]:
# Définir la fonction de perte et l'optimiseur
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print("Entraînement du modèle de base sur images étiquetées avec labels propagés...")
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True, num_workers=2)
metrics = train_and_evaluate(model, train_loader_propagated, test_loader, optimizer, criterion)

print('Entraînement terminé !')

Entraînement du modèle de base sur images étiquetées avec labels propagés...
AUC: 0.505, Accuracy: 0.347, F1: 0.322
Entraînement terminé !


### Entrainer le modéle sur différentes valeurs de K et de gamma et vois si cela a une influence significative sur notre modéle

In [ ]:
# Hyperparamètres à tester
n_neighbors_values = [5, 10, 15, 20]
gamma_values = [0.1, 1, 10, 100]

results = []

# Configuration du test_loader
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True, num_workers=2)

print("Début de la recherche de grille pour LabelSpreading...")

# --- Test du noyau 'knn' ---
for n_neighbors in tqdm(n_neighbors_values, desc="Grid Search (KNN kernel)"):
    print(f"\nTesting kernel='knn' with n_neighbors={n_neighbors}")

    # 1. Initialiser LabelSpreading
    label_spreading_model = LabelSpreading(kernel='knn', n_neighbors=n_neighbors, n_jobs=-1)

    # 2. Appliquer la propagation de labels
    label_spreading_model.fit(all_embeddings, labels_for_spreading)
    predicted_label = label_spreading_model.transduction_

    # 3. Ré-initialiser le modèle CNN
    # Note: On crée un NOUVEAU modèle et on charge ses poids originaux pour chaque test
    cnn_model = SimpleCNN(in_channels=n_channels, num_classes=n_classes)
    cnn_model.load_state_dict(torch.load(model_path, map_location="cpu"))
    cnn_model.to("cpu")
    cnn_model.eval() # Important pour la feature extraction si on refaisait des embeddings, mais ici c'est pour l'entraînement

    # Définir l'optimiseur pour ce nouveau modèle
    optimizer = torch.optim.Adam(cnn_model.parameters(), lr=0.001)

    # 4. Créer un DataLoader avec les labels propagés
    train_dataset_propagated = PropagatedDataset(train_dataset, predicted_label)
    train_loader_propagated = DataLoader(train_dataset_propagated, batch_size=32, shuffle=True, num_workers=2)

    # 5. Entraîner et évaluer le CNN
    print("Entraînement et évaluation du CNN...")
    auc, acc, f1 = train_and_evaluate(cnn_model, train_loader_propagated, test_loader, optimizer, criterion, epochs=10)

    # 6. Stocker les résultats
    results.append({
        'kernel': 'knn',
        'param_name': 'n_neighbors',
        'param_value': n_neighbors,
        'AUC': auc,
        'Accuracy': acc,
        'F1-score': f1
    })

# --- Test du noyau 'rbf' ---
for gamma in tqdm(gamma_values, desc="Grid Search (RBF kernel)"):
    print(f"\nTesting kernel='rbf' with gamma={gamma}")

    # 1. Initialiser LabelSpreading
    label_spreading_model = LabelSpreading(kernel='rbf', gamma=gamma, n_jobs=-1)

    # 2. Appliquer la propagation de labels
    label_spreading_model.fit(all_embeddings, labels_for_spreading)
    predicted_label = label_spreading_model.transduction_

    # 3. Ré-initialiser le modèle CNN
    cnn_model = SimpleCNN(in_channels=n_channels, num_classes=n_classes)
    cnn_model.load_state_dict(torch.load(model_path, map_location="cpu"))
    cnn_model.to("cpu")
    cnn_model.eval()

    # Définir l'optimiseur pour ce nouveau modèle
    optimizer = torch.optim.Adam(cnn_model.parameters(), lr=0.001)

    # 4. Créer un DataLoader avec les labels propagés
    train_dataset_propagated = PropagatedDataset(train_dataset, predicted_label)
    train_loader_propagated = DataLoader(train_dataset_propagated, batch_size=32, shuffle=True, num_workers=2)

    # 5. Entraîner et évaluer le CNN
    print("Entraînement et évaluation du CNN...")
    auc, acc, f1 = train_and_evaluate(cnn_model, train_loader_propagated, test_loader, optimizer, criterion, epochs=10)

    # 6. Stocker les résultats
    results.append({
        'kernel': 'rbf',
        'param_name': 'gamma',
        'param_value': gamma,
        'AUC': auc,
        'Accuracy': acc,
        'F1-score': f1
    })

print("\nRecherche de grille terminée !")


Début de la recherche de grille pour LabelSpreading...


Grid Search (KNN kernel):   0%|          | 0/4 [00:00<?, ?it/s]


Testing kernel='knn' with n_neighbors=5
Entraînement et évaluation du CNN...


Grid Search (KNN kernel):  25%|██▌       | 1/4 [01:26<04:19, 86.49s/it]

AUC: 0.505, Accuracy: 0.317, F1: 0.293

Testing kernel='knn' with n_neighbors=10
Entraînement et évaluation du CNN...


...

AUC: 0.517, Accuracy: 0.224, F1: 0.234

Testing kernel='rbf' with gamma=10
Entraînement et évaluation du CNN...


Grid Search (RBF kernel):  75%|███████▌  | 3/4 [03:46<01:15, 75.23s/it]

AUC: 0.532, Accuracy: 0.034, F1: 0.120

Testing kernel='rbf' with gamma=100
Entraînement et évaluation du CNN...


Grid Search (RBF kernel): 100%|██████████| 4/4 [05:00<00:00, 75.05s/it]

AUC: 0.509, Accuracy: 0.033, F1: 0.116

Recherche de grille terminée !


In [ ]:
import pandas as pd

# Afficher tous les résultats
results_df = pd.DataFrame(results)
print("\n--- Résultats de la recherche de grille ---")
display(results_df)

# Trouver la meilleure combinaison basée sur l'AUC
best_result = results_df.loc[results_df['AUC'].idxmax()]

print("\n--- Meilleure combinaison d'hyperparamètres (basée sur l'AUC) ---")
display(best_result)



--- Résultats de la recherche de grille ---


,kernel,param_name,param_value,AUC,Accuracy,F1-score
0,knn,n_neighbors,5.0,0.504672,0.316708,0.293362
1,knn,n_neighbors,10.0,0.511763,0.325686,0.318378
2,knn,n_neighbors,15.0,0.485311,0.316708,0.292819
3,knn,n_neighbors,20.0,0.514504,0.319202,0.335681
4,rbf,gamma,0.1,0.499779,0.310224,0.299126
5,rbf,gamma,1.0,0.517271,0.223940,0.233660
6,rbf,gamma,10.0,0.532008,0.033915,0.120067
7,rbf,gamma,100.0,0.508534,0.032918,0.115699



--- Meilleure combinaison d'hyperparamètres (basée sur l'AUC) ---


,6
kernel,rbf
param_name,gamma
param_value,10.0
AUC,0.532008
Accuracy,0.033915
F1-score,0.120067
